In [1]:
import pandas as pd
import numpy as np
from imblearn.under_sampling import RandomUnderSampler
from collections import defaultdict
import math

from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import cross_validate
from sklearn.metrics import recall_score, make_scorer
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.feature_selection import RFE

In [2]:
d1 = pd.read_csv("GENES\\family.csv", sep=",", dtype=str)
d2 = pd.read_csv("GENES\\gene_has_family.csv", sep=",", dtype=str)
d3 = pd.read_csv("GENES\\genes.tsv", sep="\t", dtype=str)

In [3]:
df1 = pd.DataFrame()

def Conjunto():
    caminho = input("Digite o número do conjunto de dados a ser carregado: \n\n1- Ulcerative Colitis\n2- Glioma\n3- Metastatic prostate cancer (HG-U95C)\n4- Metastatic prostate cancer (HG-U95A)\n5- Lung cancer\n6- Lung adenocarcinoma\n7- Leukemia\n8- Pulmonary hypertension\n9- Non-small cell lung carcinoma\n10- Colorectal cancer \n\n")

    global df1
    
    if caminho == '1':       # ulcerative colitis
        df = pd.read_csv("Atual\\Colitis.csv", header=None, low_memory=False)

        #Genes
        df1 = pd.read_csv("GENES\\colitis.tsv", sep="\t")
        
    elif caminho == '2':         # Glioma
        df = pd.read_csv("Atual\\Glioma.csv", header=None, low_memory=False)
        
        #Genes
        df1 = pd.read_csv("GENES\\Glioma.tsv", sep="\t")
    
    elif caminho == '3':      # Metastatic prostate cancer (HG-U95C)
        df = pd.read_csv("Atual\\Prostate1.csv", header=None, low_memory=False)
        
        # Genes
        df1 = pd.read_csv("GENES\\Prostate.tsv", sep="\t")

    elif caminho == '4':       # Metastatic prostate cancer (HG-U95A)
        df = pd.read_csv("Atual\\Prostate2.csv", header=None, low_memory=False)

        # Genes
        df1 = pd.read_csv("GENES\\Prostate.tsv", sep="\t")
        
    elif caminho == '5':   # lung cancer
        df = pd.read_csv("Atual\\Lung1.csv", header=None, low_memory=False)
    
        # Genes
        df1 = pd.read_csv("GENES\\LungCarcinoma.tsv", sep="\t")
        
    elif caminho == '6':       # Lung adenocarcinoma
        df = pd.read_csv("Atual\\Lung2.csv", header=None, low_memory=False)
        
        # Genes
        df1 = pd.read_csv("GENES\\AdenocarcinomaLung.tsv", sep="\t")
        
    elif caminho == '7':       # Leukemia
        df = pd.read_csv("Atual\\Leukemia.csv", header=None, low_memory=False)
        
        # Genes
        df1 = pd.read_csv("GENES\\leukemia.tsv", sep="\t")
    
    elif caminho == '8':      # Pulmonary hypertension
        df = pd.read_csv("Atual\\Pulmonary.csv", header=None, low_memory=False)
        
        # Genes
        df1 = pd.read_csv("GENES\\pulmonary.tsv", sep="\t")
        
    elif caminho == '9':       # Non-small cell lung carcinoma
        df = pd.read_csv("Atual\\nonsmall.csv", header=None, low_memory=False)

        # Genes
        df1 = pd.read_csv("GENES\\non-small.tsv", sep="\t")
        
    elif caminho == '10':     # Colorectal cancer
        df = pd.read_csv("Atual\\Colorectal.csv", header=None, low_memory=False)
        
        # Genes
        df1 = pd.read_csv("GENES\\colorectal.tsv", sep="\t")
        
    else:
        return 0
    
    # Resolve o problema do pandas identificar classes iguais e mudar o seu nome. 
    novo = df.iloc[0]
    df.columns = novo
    df = df.drop(0).reset_index(drop=True)
    
    return df

In [4]:
clas = []

def SubConjuntos():
    conjunto = Conjunto()
    
    if isinstance(conjunto, int):
        conjunto = Conjunto()

    global clas
    
    ent = input("Deseja utilizar o conjunto de dados original? (y/n)")

    if ent == 'n':   
        # Redução de Dimensionalidade pela base da DisgenNET
        arr = []
    
        for i in df1['Gene']:
            if i in conjunto.columns:
                arr.append(i)
    
        arr.append('Classe')
    
        fim = conjunto.loc[:, arr]
    
        print("Conjunto reduzido:\n")
        print(fim)
        
        clas = fim['Classe'].astype('int')
        fim = fim.drop('Classe', axis=1)
        
        # Gerar as Familias
        result = pd.merge(d2, d3, on=['hgnc_id'], how="inner")
        result = result.loc[:, ['family_id', 'symbol']].astype({'family_id': int, 'symbol': str}) 
        
        r = result[result['symbol'].isin(fim.columns)]
        r = r.sort_values('family_id')
        
        familias = []                 # ← lista final de dicionários
        genes = []                 # ← acumula símbolos do bloco atual
        fam_atual = None              # ← sentinel para detectar a 1ª linha
    
        for _, linha in r.iterrows():      # r já está em ordem por family_id
            fid, gene = linha['family_id'], linha['symbol']
    
            # Se mudou a família, salva o bloco anterior e zera o acumulador
            if fid != fam_atual and fam_atual is not None:
                familias.append({fam_atual: genes})
                genes = []                 # reinicia a lista para o próximo bloco
    
            # Continua no mesmo (ou inicia o novo) bloco
            genes.append(gene)
            fam_atual = fid
    
        # Depois do laço, grava o último bloco
        if genes:
            familias.append({fam_atual: genes})
            
        # Gera uma lista de subconjuntos passando o id da família para cada "nome" do subconjunto
        listaa = [(_df := fim.loc[:, next(iter(d.values()))]).attrs.__setitem__("Família", next(iter(d.keys()))) or _df for d in familias]
        
        # subdivir subconjuntos grandes
        med = 0
    
        for i in listaa:
            med = med + len(i.columns)
    
        me = med / len(listaa)
        me = round(me)
        
        lista = []
    
        for i in listaa:
            if len(i.columns) > me:
                x = len(i.columns) / me
                count = 0
                for d in range(math.ceil(x)):
                    lista.append(i.iloc[:,count:count+me].astype('float'))
                    count = count + me
            else:
                lista.append(i.astype('float'))
                
        print('Quantidade de subconjuntos gerados: ', len(lista))
    
        return lista

    else:
        clas = conjunto['Classe'].astype('int')

        if '--Control' in conjunto.columns:
            conjunto = conjunto.drop('--Control', axis=1).astype('float')
            
        if 'BrightCorner' in conjunto.columns:
            conjunto = conjunto.drop("BrightCorner", axis=1).astype('float')
        
        print("Conjunto de dados Original: \n")
        print(conjunto)
    
        conjunto = conjunto.drop('Classe', axis=1)

        entr = int(input("\n1- Normal\n2- ANOVA\n3- Mutual\n4- Sequential\n"))

        global att, seq
        seq = False
        att = 0
        
        if entr == 1: # Conjundo de dados original
            return conjunto

        elif entr == 2: # ANOVA 5, 10 , 20 e 40.
            att = int(input('\nQuantidade de atributos: '))
            X_new = SelectKBest(f_classif, k=att) # k = quantidade de atributos para selecionar
            nov = X_new.fit_transform(conjunto, clas)
            conjunto = pd.DataFrame(nov)
            return conjunto

        else:: # Mutual
            att = int(input('\nQuantidade de atributos: '))
            mutual = partial(mutual_info_classif, random_state=42)
            X_new = SelectKBest(mutual, k=att) # k = quantidade de atributos para selecionar
            nov = X_new.fit_transform(conjunto, clas)
            conjunto = pd.DataFrame(nov)
            return conjunto

In [ ]:
def Modelo():
    lista = SubConjuntos()

    #Ranquear os subconjuntos
    
    cvr = ShuffleSplit(n_splits=100, test_size=0.25, random_state=42) # divisões para os subconjuntos
    cvm = ShuffleSplit(n_splits=100, test_size=0.25, random_state=41) # divisão para o modelo final
    
    resul = defaultdict(list)
    
    ml = input("Digite o algoritmo: \n\n1- Random Forest\n2 - HistGBM\n3- GB\n4- KNN\n5- SVM\n6- NaiveBayes\n")
    
    if ml == '1':
        model = RandomForestClassifier(n_estimators=50, random_state=42)
      
    elif ml == '2':      
        model = HistGradientBoostingClassifier(max_iter=50, min_samples_leaf=6, random_state=42)

    elif ml == '3':      
        model = GradientBoostingClassifier(n_estimators=50, random_state=42)
    
    elif ml == '4':
        model = KNeighborsClassifier(n_neighbors=17)

    elif ml == '5':
        model = LinearSVC(max_iter=100000, random_state=42)

    elif ml == '6':
        model = GaussianNB()

    sensit = make_scorer(recall_score, pos_label=1)
    specif = make_scorer(recall_score, pos_label=0)
    scorers = {'accuracy': 'accuracy', 'roc_auc': 'roc_auc', 'f1': 'f1', 'precision': 'precision', 'sensit': sensit, 'specif': specif}
    
    if isinstance(lista, list):   
        for idx, i in enumerate(lista):
            scores = cross_validate(model, i, clas,scoring='accuracy', cv=cvr, n_jobs=-1)
    
            resul[idx].append(scores['test_score'].mean())
        
        ordenados = sorted(resul.items(), key=lambda item: item[1], reverse=True)
        ordenados = dict(ordenados)
    
        # Modelo Final
        # Para o conjunto 10 foi verificado o menor numero de atributos que da a acurácia de 100% e foi com apenas 1 atributo.
                    
        count = 1 # Top 10
        iguais = []
        gen = []
        
        for chave, valor in ordenados.items():
            if count < 11:
                if count == 1:
                    print(count,"° do ranking e Grupo", lista[chave].attrs, 'Genes: ',lista[chave].columns.to_list(),'\n')
    
                    scor = cross_validate(model, lista[chave], clas, scoring=scorers, cv=cvr, n_jobs=-1)
                    m_auc = scor['test_roc_auc'].mean()
                    m_f1 = scor['test_f1'].mean()
                    m_prec = scor['test_precision'].mean()
                    m_sensit = scor['test_sensit'].mean()
                    m_specif = scor['test_specif'].mean()
                    
                    print('ACC: ', valor[0], '\nAUC: ', m_auc, '\nF1: ', m_f1, '\nPrecision: ', m_prec, '\nSensitivity: ', m_sensit, '\nSpecificity: ', m_specif, "\n\n")
    
                    anterior = lista[chave]
                    count = count + 1
    
                else: 
                    print(count,"° do ranking: ",valor[0], lista[chave].attrs,'Genes: ',lista[chave].columns.to_list(),'\n')
    
                    aux = pd.concat([anterior, lista[chave]], axis=1)
    
                    # Verificar se há duplicados
                    for i in range(0,len(aux.columns)):
                        for j in range(0,len(aux.columns)):
                            if i != j:
                                if (aux.iloc[:, i] == aux.iloc[:, j]).all():
                                    iguais.append(aux.columns[i])
    
                    aux = aux.T.drop_duplicates(keep='first').T
                
                    scor = cross_validate(model, aux, clas, scoring=scorers, cv=cvm, n_jobs=-1)
                    m_acc = scor['test_accuracy'].mean()
                    m_auc = scor['test_roc_auc'].mean()
                    m_f1 = scor['test_f1'].mean()
                    m_prec = scor['test_precision'].mean()
                    m_sensit = scor['test_sensit'].mean()
                    m_specif = scor['test_specif'].mean()
    
                    print(count,'° Grupo -', " Quantidade de atributos: ", len(aux.columns),"\n")
                    print('ACC: ', m_acc, '\nAUC: ', m_auc, '\nF1: ', m_f1, '\nPrecision: ', m_prec, '\nSensitivity: ', m_sensit, '\nSpecificity: ', m_specif, "\n\n")
    
                    anterior = aux
                    count = count + 1
                
            else:
                break
            
        print('Genes iguais: ', set(iguais))

    else:
        if seq == True:
            selector = RFE(model, n_features_to_select=att, step=1)
            dat = selector.fit_transform(lista, clas)
            lista = pd.DataFrame(dat)

        
        scor = cross_validate(model, lista, clas, scoring=scorers, cv=cvm, n_jobs=-1)
        m_acc = scor['test_accuracy'].mean()
        m_auc = scor['test_roc_auc'].mean()
        m_f1 = scor['test_f1'].mean()
        m_prec = scor['test_precision'].mean()
        m_sensit = scor['test_sensit'].mean()
        m_specif = scor['test_specif'].mean()

        print('\nMétricas: \n\nACC: ', m_acc, '\nAUC: ', m_auc, '\nF1: ', m_f1, '\nPrecision: ', m_prec, '\nSensitivity: ', m_sensit, '\nSpecificity: ', m_specif, "\n\n")

In [ ]:
def Converter():
    fami = input("Digite o ID da família ou 0 para sair: \n")
    
    if fami == '0':
        exit()
        
    else:    
        identif = d1.loc[d1['id'] == fami]
        identif = identif.iat[0,2]
    
        print('Nome da Família: ', identif)
        
        Converter()

In [ ]:
if __name__ == "__main__":
    
    digi = input('Digite o número correspondente: \n\n1 - Carregar conjuntos\n2- Converter Famílias\n')
    
    if digi == '1':
        Modelo()
        
    elif digi == '2':
        Converter()
        
    else:
        exit()